# Orchestration Eval Analysis Template

Analyzes a committed run of the offline adversarial harness
(`agent-orchestration-offline` suite). Regenerate the artifact first:

```
pnpm run fde:evals
```

This notebook is read-only over repo artifacts and uses only the Python
standard library.

In [ ]:
import json
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
summary = json.loads((REPO_ROOT / "docs" / "benchmark" / "offline-eval-summary.json").read_text())
print(f"suite={summary['suite']} passed={summary['passed']} total={summary['total']} failed={summary['failed']}")

## Per-fixture pass matrix

`expectedPass` is the fixture's design intent; `observedPass` is what the
Evaluator saw. A healthy suite has `passed=True` on every row — including
the adversarial rows, where passing means the violation *was caught*.

In [ ]:
header = f"{'fixture':40} {'expected':8} {'observed':8} {'passed':6}"
print(header)
print("-" * len(header))
for result in summary["results"]:
    print(f"{result['id']:40} {str(result['expectedPass']):8} {str(result['observedPass']):8} {str(result['passed']):6}")

## Four-axis score breakdown

Correctness / Safety / Completeness / Quality per fixture, plus suite means.
Adversarial fixtures are expected to score low on the axis they attack.

In [ ]:
import statistics

axes = ["correctness", "safety", "completeness", "quality"]
for result in summary["results"]:
    scores = " ".join(f"{axis[0].upper()}={result['scores'][axis]:.2f}" for axis in axes)
    print(f"{result['id']:40} {scores}")
print()
for axis in axes:
    values = [result["scores"][axis] for result in summary["results"]]
    print(f"mean {axis:12} {statistics.mean(values):.3f}")

## Issue codes on adversarial fixtures

Every negative control should report the violation class it was built to
trigger. Empty issue lists on a `Fail`-expected fixture mean the Evaluator
has a blind spot.

In [ ]:
for result in summary["results"]:
    if result["expectedPass"]:
        continue
    print(result["id"])
    for issue in result["issues"]:
        print(f"  - {issue}")

## Plateau inspection across runs

To study score movement over time, collect `mean completeness` (or any
scalar) from successive summaries and check when the series flattens —
the same logic `createPlateauTracker` applies in `packages/fde`
(`window` of 3, `epsilon` of 0.01).

In [ ]:
def plateaued(series, window=3, epsilon=0.01):
    recent = series[-window:]
    return len(recent) >= window and (max(recent) - min(recent)) <= epsilon

# Replace with scalars gathered from successive eval runs.
example_series = [0.72, 0.81, 0.83, 0.83, 0.84]
print(f"series={example_series} plateaued={plateaued(example_series)}")